# Modelo de Classificação: Random Forest para Predição de Popularidade

 Este script documenta o desenvolvimento de um algoritmo de Aprendizado de
 Máquina Supervisionado (Random Forest) com o objetivo de classificar o
 potencial de popularidade de obras literárias no Booklog em 3 classes:
   Classe 1 → Bestseller     (≥ 10.000 avaliações)
   Classe 2 → Média Pop.     (≥ 1.000 avaliações)
   Classe 3 → Nicho          (< 1.000 avaliações)


Preparação dos dados

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import GridSearchCV
import shap
import numpy as np
import joblib
import os


df_rf = pd.read_parquet('../data/processed/books_pivot_mapped.parquet')
 
# 1.2 Criando o Alvo triclasse
def definir_tres_classes(ratings):
    if ratings >= 10000:
        return 1   # Bestseller
    elif ratings >= 1000:
        return 2   # Média Popularidade
    else:
        return 3   # Nicho
 
df_rf['popularity_class'] = df_rf['totalratings'].apply(definir_tres_classes)
 
print("=" * 60)
print("DISTRIBUIÇÃO ORIGINAL DAS CLASSES (antes do balanceamento)")
print("=" * 60)
dist = df_rf['popularity_class'].value_counts().sort_index()
total = len(df_rf)
for classe, qtd in dist.items():
    nomes = {1: "Bestseller", 2: "Média Popularidade", 3: "Nicho"}
    print(f"  Classe {classe} ({nomes[classe]}): {qtd:,} livros ({qtd/total*100:.1f}%)")
print(f"  Total: {total:,} livros")
print("-" * 60)
 
# 1.3 Separando preditores e alvo

colunas_proibidas = ['title', 'author', 'rating', 'totalratings', 'popularity_class']
X = df_rf.drop(columns=colunas_proibidas)
y = df_rf['popularity_class']
 
# 1.4 Divisão Treino/Teste — stratify garante proporção de classes em ambos os conjuntos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
 
# ──author_frequency calculado APENAS sobre o treino 

autor_frequencia = df_rf.loc[X_train.index, 'author'].value_counts()
 
X_train = X_train.copy()
X_test  = X_test.copy()
 
X_train['author_frequency'] = df_rf.loc[X_train.index, 'author'].map(autor_frequencia).fillna(0)
# Autores do teste não vistos no treino recebem 0 (autor desconhecido)
X_test['author_frequency']  = df_rf.loc[X_test.index,  'author'].map(autor_frequencia).fillna(0)
 
print("\nEngenharia de frequência de autores aplicada sem data leakage.")
print(f"  Autores únicos no treino: {autor_frequencia.shape[0]:,}")
print(f"  Autores do teste não vistos no treino: {X_test['author_frequency'].eq(0).sum()} livros → recebem 0\n")


DISTRIBUIÇÃO ORIGINAL DAS CLASSES (antes do balanceamento)
  Classe 1 (Bestseller): 3,688 livros (4.5%)
  Classe 2 (Média Popularidade): 16,649 livros (20.3%)
  Classe 3 (Nicho): 61,642 livros (75.2%)
  Total: 81,979 livros
------------------------------------------------------------

Engenharia de frequência de autores aplicada sem data leakage.
  Autores únicos no treino: 42,148
  Autores do teste não vistos no treino: 15015 livros → recebem 0



## 2. Treinamento inicial(balanceamento nativo)

In [34]:
print("Iniciando Random Forest com balanceamento nativo (class_weight='balanced')...")
 
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)
previsoes = rf_model.predict(X_test)
 
print("=" * 70)
print("  RESULTADOS — RANDOM FOREST COM BALANCEAMENTO NATIVO")
print("=" * 70)
print(f"Acurácia Global: {accuracy_score(y_test, previsoes) * 100:.2f}%\n")
print(classification_report(
    y_test, previsoes,
    target_names=['Bestseller', 'Média Popularidade', 'Nicho']
))


Iniciando Random Forest com balanceamento nativo (class_weight='balanced')...
  RESULTADOS — RANDOM FOREST COM BALANCEAMENTO NATIVO
Acurácia Global: 72.05%

                    precision    recall  f1-score   support

        Bestseller       0.22      0.24      0.23      1106
Média Popularidade       0.41      0.44      0.43      4995
             Nicho       0.85      0.82      0.83     18493

          accuracy                           0.72     24594
         macro avg       0.49      0.50      0.50     24594
      weighted avg       0.73      0.72      0.72     24594



## 3. Undersampling fisico
 Aplicamos undersampling no treino para comparação metodologicamente justa
 com os demais modelos do projeto. O conjunto de teste NÃO é alterado,
 garantindo avaliação na distribuição real dos dados.


In [35]:
print("Iniciando a Otimização Científica da Random Forest...")

# 1. As configurações que queremos testar
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10]
}

# 2. Configurando o Buscador
rf_base = RandomForestClassifier(random_state=42)

grid_search_rf = GridSearchCV(
    estimator=rf_base, 
    param_grid=param_grid, 
    scoring='f1_macro', 
    cv=5,               
    verbose=1, 
    n_jobs=-1           
)

# 3. Treinando com os dados reduzidos e balanceados
grid_search_rf.fit(X_train_resampled, y_train_resampled)

# 4. Extraindo o modelo Campeão
print("\n==================================================")
print("          Teste com a melhor floresta encontrada")
print("==================================================")
print(f"Melhores parâmetros: {grid_search_rf.best_params_}")
print(f"F1-Score Médio de Validação: {grid_search_rf.best_score_ * 100:.2f}%")
print("==================================================\n")

# 5. Testando o modelo
melhor_rf = grid_search_rf.best_estimator_
previsoes_otimizadas = melhor_rf.predict(X_test)

print("Relatório de Classificação Final do RF Otimizado Undersampling:")
print("-" * 70)
print(f"Acurácia Global no Teste: {accuracy_score(y_test, previsoes_otimizadas) * 100:.2f}%\n")
print("-" * 70)
print(classification_report(y_test, previsoes_otimizadas, target_names=['Classe 1', 'Classe 2', 'Classe 3']))

Iniciando a Otimização Científica da Random Forest...
Fitting 5 folds for each of 27 candidates, totalling 135 fits

          Teste com a melhor floresta encontrada
Melhores parâmetros: {'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 100}
F1-Score Médio de Validação: 59.49%

Relatório de Classificação Final do RF Otimizado Undersampling:
----------------------------------------------------------------------
Acurácia Global no Teste: 65.60%

----------------------------------------------------------------------
              precision    recall  f1-score   support

    Classe 1       0.18      0.54      0.27      1106
    Classe 2       0.35      0.48      0.40      4995
    Classe 3       0.91      0.71      0.80     18493

    accuracy                           0.66     24594
   macro avg       0.48      0.58      0.49     24594
weighted avg       0.76      0.66      0.69     24594



# 4 Otimização com GridSearchCV
O GridSearchCV faz cross-validation internamente nos dados
 undersampled. O F1-Score de validação reportado representa a distribuição
 artificial balanceada — não a distribuição real. Por isso, a avaliação final
 sempre é feita no X_test original (não balanceado).


In [36]:
print("Iniciando otimização com GridSearchCV (cv=5, scoring=f1_macro)...")
 
param_grid = {
    'n_estimators':    [100, 200, 300],
    'max_depth':       [10, 20, None],
    'min_samples_split': [2, 5, 10],
}
 
rf_base = RandomForestClassifier(random_state=42)
 
grid_search_rf = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=5,
    verbose=1,
    n_jobs=-1,
)
 
grid_search_rf.fit(X_train_resampled, y_train_resampled)
 
print("\n" + "=" * 60)
print("  MELHOR CONFIGURAÇÃO ENCONTRADA")
print("=" * 60)
print(f"Melhores parâmetros: {grid_search_rf.best_params_}")
print(f"F1-Score Médio de Validação (dados balanceados): {grid_search_rf.best_score_ * 100:.2f}%")
print("  ⚠ Este F1 é calculado sobre dados undersampled (distribuição artificial).")
print("  A avaliação real está no relatório abaixo (conjunto de teste original).")
print("=" * 60 + "\n")
 
melhor_rf = grid_search_rf.best_estimator_
previsoes_otimizadas = melhor_rf.predict(X_test)
 
print("Relatório Final — RF Otimizado — conjunto de TESTE (distribuição real):")
print("-" * 70)
print(f"Acurácia Global no Teste: {accuracy_score(y_test, previsoes_otimizadas) * 100:.2f}%\n")
print(classification_report(
    y_test, previsoes_otimizadas,
    target_names=['Bestseller', 'Média Popularidade', 'Nicho']
))
 
previsoes_treino_original = melhor_rf.predict(X_train)
 
print("=" * 70)
print("  DIAGNÓSTICO DE OVERFITTING")
print("  Avaliação no X_train ORIGINAL (não balanceado)")
print("=" * 70)
print(f"Acurácia no Treino (original): {accuracy_score(y_train, previsoes_treino_original) * 100:.2f}%")
print(f"Acurácia no Teste:             {accuracy_score(y_test,  previsoes_otimizadas)       * 100:.2f}%")
gap = accuracy_score(y_train, previsoes_treino_original) - accuracy_score(y_test, previsoes_otimizadas)
print(f"Gap Treino-Teste:              {gap * 100:.2f}pp  {'(overfitting moderado)' if gap > 0.10 else '(aceitável)'}")
print()
print(classification_report(
    y_train, previsoes_treino_original,
    target_names=['Bestseller', 'Média Popularidade', 'Nicho']
))


Iniciando otimização com GridSearchCV (cv=5, scoring=f1_macro)...
Fitting 5 folds for each of 27 candidates, totalling 135 fits

  MELHOR CONFIGURAÇÃO ENCONTRADA
Melhores parâmetros: {'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 100}
F1-Score Médio de Validação (dados balanceados): 59.49%
  ⚠ Este F1 é calculado sobre dados undersampled (distribuição artificial).
  A avaliação real está no relatório abaixo (conjunto de teste original).

Relatório Final — RF Otimizado — conjunto de TESTE (distribuição real):
----------------------------------------------------------------------
Acurácia Global no Teste: 65.60%

                    precision    recall  f1-score   support

        Bestseller       0.18      0.54      0.27      1106
Média Popularidade       0.35      0.48      0.40      4995
             Nicho       0.91      0.71      0.80     18493

          accuracy                           0.66     24594
         macro avg       0.48      0.58      0.49     24594
      w

## 5. Componentes para Produção
Para integração com o back-end do aplicativo, exportamos o modelo treinado e os dicionários auxiliares. Estes arquivos permitirão que o sistema receba dados de um novo livro e devolva sua classificação preditiva em tempo real.

In [37]:
caminho_modelos = '../../Machine Learning/models'
os.makedirs(caminho_modelos, exist_ok=True)
 
joblib.dump(melhor_rf,       os.path.join(caminho_modelos, 'RF_popularidade.pkl'))
joblib.dump(autor_frequencia, os.path.join(caminho_modelos, 'autor_frequencia_RF.pkl'))
 
print("Artefatos exportados para produção com sucesso.")


Artefatos exportados para produção com sucesso.


# 6. Exportação do SHAP para uso no dashboard

In [38]:
explainer = shap.TreeExplainer(melhor_rf)
X_sample  = X_test.sample(min(1000, len(X_test)), random_state=42)
shap_values = explainer(X_sample)
 
feat_df   = X_sample.reset_index(drop=True)
 
# Normalização das features para cor no beeswarm
feat_norm = pd.DataFrame()
for col in feat_df.columns:
    if feat_df[col].nunique() <= 2:
        # Binária (gêneros): normalização min-max simples
        mn, mx = feat_df[col].min(), feat_df[col].max()
        feat_norm[col] = (feat_df[col] - mn) / (mx - mn) if mx > mn else 0.0
    else:
        # Contínua (pages, author_frequency): rank percentual
        feat_norm[col] = feat_df[col].rank(pct=True)
 
# Mapa de classes para exportação
classes_shap = {
    0: {'idx': 0, 'nome': 'Bestseller',          'arquivo': 'shap_beeswarm_classe1_bestseller.parquet'},
    1: {'idx': 1, 'nome': 'Media_Popularidade',  'arquivo': 'shap_beeswarm_classe2_media.parquet'},
    2: {'idx': 2, 'nome': 'Nicho',               'arquivo': 'shap_beeswarm_classe3_nicho.parquet'},
}
 
caminho_processed = '../data/processed'
 
for k, cfg in classes_shap.items():
    vals = shap_values.values[:, :, cfg['idx']]
 
    shap_df = pd.DataFrame(vals, columns=X_sample.columns)
 
    melted_shap = shap_df.melt(var_name='Feature', value_name='SHAP_Value')
    melted_feat = feat_norm.melt(var_name='Feature', value_name='Feature_Value')
 
    beeswarm_data = pd.DataFrame({
        'Feature':       melted_shap['Feature'],
        'SHAP_Value':    melted_shap['SHAP_Value'],
        'Feature_Value': melted_feat['Feature_Value'].clip(0, 1),
    })
 
    # Ordenação por importância média absoluta (mais importante no topo)
    media_importancia = (
        beeswarm_data
        .assign(Abs_SHAP=lambda d: d['SHAP_Value'].abs())
        .groupby('Feature')['Abs_SHAP']
        .mean()
        .sort_values(ascending=True)   # ascending=True → menor importância embaixo
    )
 
    beeswarm_data['Feature'] = pd.Categorical(
        beeswarm_data['Feature'],
        categories=media_importancia.index,
        ordered=True
    )
 
    caminho_saida = os.path.join(caminho_processed, cfg['arquivo'])
    beeswarm_data.to_parquet(caminho_saida)
    print(f"SHAP exportado → {cfg['arquivo']}")
 
print("\nTodos os artefatos SHAP exportados com sucesso.")

SHAP exportado → shap_beeswarm_classe1_bestseller.parquet
SHAP exportado → shap_beeswarm_classe2_media.parquet
SHAP exportado → shap_beeswarm_classe3_nicho.parquet

Todos os artefatos SHAP exportados com sucesso.
